# Distributed Trigger-Based Aggregation (Citus)

Compare two approaches for maintaining a confusion-matrix aggregation over
`pred_patch` × `patch`:

| Approach | Mechanism |
|---|---|
| **Trigger** | Per-shard statement-level triggers incrementally maintain `v4_confusion_matrix_partial`: `AFTER INSERT` on `v4_pred_patch` adds counts; `AFTER UPDATE` on `v4_patch` adjusts counts when `gt_label` changes (negative deltas for old label, positive for new) |
| **Materialized View** | `REFRESH MATERIALIZED VIEW` over a `GROUP BY` join |

Both tables (`v4_patch`, `v4_pred_patch`) are distributed on `id`.
`v4_confusion_matrix_partial` is distributed on `shard_id` (Citus shard ID).
Grid level: **l12** (finest, 4096×4096).

In [ ]:
import psycopg2

In [ ]:
DATABASE_URL = "dbname=citus user=citus password=citus host=localhost port=5439"
TABLE_PREFIX = "v4"
PATCH_TABLE = f"{TABLE_PREFIX}_patch"
PRED_PATCH_TABLE = f"{TABLE_PREFIX}_pred_patch"
CM_TABLE = f"{TABLE_PREFIX}_confusion_matrix_partial"
MATVIEW_NAME = f"{TABLE_PREFIX}_patch_label_agg_l12"

## 1 – Drop existing tables and views

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"DROP MATERIALIZED VIEW IF EXISTS {MATVIEW_NAME} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {CM_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PRED_PATCH_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PATCH_TABLE} CASCADE;")

conn.commit()
cur.close()
conn.close()

print(f"Dropped (if existed): {MATVIEW_NAME}, {CM_TABLE}, {PRED_PATCH_TABLE}, {PATCH_TABLE}")

## 2 – Create `v4_patch` (distributed on `id`)

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE UNLOGGED TABLE {PATCH_TABLE}(
    id BIGINT NOT NULL,
    patch_uid integer NOT NULL,
    gt_label integer,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    image_id integer,
    working_mag double precision,
    PRIMARY KEY(id)
);
""")
cur.execute(f"SELECT create_distributed_table('{PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PATCH_TABLE} on 'id'.")

## 3 – Create `v4_pred_patch` (distributed on `id`)

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE UNLOGGED TABLE {PRED_PATCH_TABLE}(
    id BIGINT NOT NULL,
    patch_uid bigint NOT NULL,
    embed_coords point,
    grid_cell_i int,
    grid_cell_j int,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    pred_label integer,
    patch_coords point,
    PRIMARY KEY(id)
);

CREATE INDEX idx_{PRED_PATCH_TABLE}_grid_cells
    ON {PRED_PATCH_TABLE} (grid_cell_i, grid_cell_j);
""")
cur.execute(f"SELECT create_distributed_table('{PRED_PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PRED_PATCH_TABLE} on 'id'.")

## 4 – Create `v4_confusion_matrix_partial` (distributed on `shard_id`)

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE TABLE {CM_TABLE} (
    shard_id    BIGINT NOT NULL,
    grid_i      INT NOT NULL,
    grid_j      INT NOT NULL,
    pred_label  INT NOT NULL,
    gt_label    INT NOT NULL,
    cnt         BIGINT DEFAULT 0,
    PRIMARY KEY (shard_id, grid_i, grid_j, pred_label, gt_label)
);
""")
# Colocate with pred_patch so shard-level triggers can do local joins
cur.execute(f"""
    SELECT create_distributed_table(
        '{CM_TABLE}', 'shard_id',
        colocate_with => '{PRED_PATCH_TABLE}'
    );
""")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {CM_TABLE} on 'shard_id', colocated with {PRED_PATCH_TABLE}.")

## 5 – Per-shard statement-level triggers

Citus doesn't support triggers on distributed tables directly, but we can
create them on individual **shard placements** using
`run_command_on_colocated_placements`.

### Trigger A – `AFTER INSERT` on `v4_pred_patch`

Each shard of `v4_pred_patch` gets an `AFTER INSERT … FOR EACH STATEMENT`
trigger with a `REFERENCING NEW TABLE AS new_rows` transition table.
The trigger function (`update_cm_shard`):
1. Derives the colocated `v4_patch` shard name from `pg_dist_shard` metadata
2. JOINs `new_rows` with the local `v4_patch` shard to get `gt_label`
3. Pre-aggregates via `GROUP BY` and upserts into the colocated
   `v4_confusion_matrix_partial` shard

### Trigger B – `AFTER UPDATE` on `v4_patch`

Each shard of `v4_patch` gets an `AFTER UPDATE … FOR EACH STATEMENT`
trigger with `REFERENCING OLD TABLE AS old_rows NEW TABLE AS new_rows`.
The trigger function (`update_cm_on_patch_update`):
1. Identifies rows where `gt_label` actually changed (`IS DISTINCT FROM`)
2. JOINs with the colocated `v4_pred_patch` shard to find affected predictions
3. Computes **negative deltas** (old gt_label) and **positive deltas** (new gt_label)
4. Nets the deltas and upserts into the CM shard (`cnt` can decrease)
5. Cleans up any CM rows where `cnt` has reached 0

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# ---------------------------------------------------------------------------
# Step 1a: Create the INSERT trigger function (pred_patch → CM).
#          Citus 13 DDL propagation pushes it to workers automatically;
#          we also explicitly run it on workers as a safety net.
#
# NOTE: TG_ARGV[0] arrives from run_command_on_colocated_placements as a
# schema-qualified name (e.g. "public.v4_confusion_matrix_partial_102040").
# We must use %s (raw) rather than %I (identifier-quoted) so that the dot is
# parsed as a schema separator, not embedded in the identifier.
# ---------------------------------------------------------------------------
trigger_fn_sql = f"""
CREATE OR REPLACE FUNCTION update_cm_shard()
RETURNS TRIGGER LANGUAGE plpgsql AS $body$
DECLARE
    v_patch_shard text;
    v_pred_shardid bigint;
BEGIN
    -- Extract the numeric shard ID from the current shard table name
    -- e.g. 'v4_pred_patch_102008' -> 102008
    v_pred_shardid := (regexp_match(TG_TABLE_NAME, '(\d+)$'))[1]::bigint;

    -- Resolve the colocated v4_patch shard (same hash-range).
    -- Build as schema.table using the current trigger's schema (TG_TABLE_SCHEMA).
    SELECT TG_TABLE_SCHEMA || '.' || '{PATCH_TABLE}_' || shardid::text
      INTO v_patch_shard
    FROM pg_dist_shard
    WHERE logicalrelid = '{PATCH_TABLE}'::regclass
      AND shardminvalue = (
          SELECT shardminvalue FROM pg_dist_shard
          WHERE shardid = v_pred_shardid
      );

    -- TG_ARGV[0] = colocated confusion_matrix_partial shard (schema-qualified)
    -- Use %s (not %I) for schema-qualified names so the dot is a schema separator
    EXECUTE format(
        'INSERT INTO %s (shard_id, grid_i, grid_j, pred_label, gt_label, cnt) '
        'SELECT 0, nr.grid_cell_i, nr.grid_cell_j, nr.pred_label, '
        '       COALESCE(p.gt_label, -1), COUNT(*) '
        'FROM new_rows nr '
        'LEFT JOIN %s p ON nr.id = p.id '
        'GROUP BY nr.grid_cell_i, nr.grid_cell_j, nr.pred_label, p.gt_label '
        'ON CONFLICT (shard_id, grid_i, grid_j, pred_label, gt_label) '
        'DO UPDATE SET cnt = EXCLUDED.cnt + %s.cnt',
        TG_ARGV[0], v_patch_shard, TG_ARGV[0]
    );
    RETURN NULL;
END;
$body$;
"""

cur.execute(trigger_fn_sql)
conn.commit()
cur.execute(f"SELECT run_command_on_workers($outer${trigger_fn_sql}$outer$);")
conn.commit()
print("INSERT trigger function created on coordinator and workers.")

# ---------------------------------------------------------------------------
# Step 1b: Create the UPDATE trigger function (patch gt_label change → CM).
#
# When gt_label changes on v4_patch rows, we:
#   1. Find all v4_pred_patch rows that reference the changed patch IDs
#   2. Subtract counts for the OLD gt_label (negative delta)
#   3. Add counts for the NEW gt_label (positive delta)
#   4. Net the deltas and upsert into the colocated CM shard
#   5. Delete any CM rows that have been decremented to cnt = 0
# ---------------------------------------------------------------------------
update_trigger_fn_sql = f"""
CREATE OR REPLACE FUNCTION update_cm_on_patch_update()
RETURNS TRIGGER LANGUAGE plpgsql AS $body$
DECLARE
    v_pred_shard text;
    v_patch_shardid bigint;
BEGIN
    -- Extract shard ID from the patch shard table name
    v_patch_shardid := (regexp_match(TG_TABLE_NAME, '(\d+)$'))[1]::bigint;

    -- Resolve colocated v4_pred_patch shard (same hash-range)
    SELECT TG_TABLE_SCHEMA || '.' || '{PRED_PATCH_TABLE}_' || shardid::text
      INTO v_pred_shard
    FROM pg_dist_shard
    WHERE logicalrelid = '{PRED_PATCH_TABLE}'::regclass
      AND shardminvalue = (
          SELECT shardminvalue FROM pg_dist_shard
          WHERE shardid = v_patch_shardid
      );

    -- TG_ARGV[0] = colocated CM shard (schema-qualified)
    -- Compute negative deltas (old gt_label) and positive deltas (new gt_label),
    -- net them, and upsert.  Only process rows where gt_label actually changed.
    EXECUTE format(
        'WITH changed AS ( '
        '    SELECT o.id, '
        '           COALESCE(o.gt_label, -1) AS old_gt, '
        '           COALESCE(n.gt_label, -1) AS new_gt '
        '    FROM old_rows o '
        '    JOIN new_rows n ON o.id = n.id '
        '    WHERE o.gt_label IS DISTINCT FROM n.gt_label '
        '), '
        'neg AS ( '
        '    SELECT pp.grid_cell_i, pp.grid_cell_j, pp.pred_label, '
        '           c.old_gt AS gt_label, -COUNT(*)::bigint AS delta '
        '    FROM changed c '
        '    JOIN %s pp ON pp.id = c.id '
        '    GROUP BY pp.grid_cell_i, pp.grid_cell_j, pp.pred_label, c.old_gt '
        '), '
        'pos AS ( '
        '    SELECT pp.grid_cell_i, pp.grid_cell_j, pp.pred_label, '
        '           c.new_gt AS gt_label, COUNT(*)::bigint AS delta '
        '    FROM changed c '
        '    JOIN %s pp ON pp.id = c.id '
        '    GROUP BY pp.grid_cell_i, pp.grid_cell_j, pp.pred_label, c.new_gt '
        '), '
        'deltas AS ( '
        '    SELECT grid_cell_i, grid_cell_j, pred_label, gt_label, '
        '           SUM(delta) AS net_delta '
        '    FROM ( SELECT * FROM neg UNION ALL SELECT * FROM pos ) t '
        '    GROUP BY grid_cell_i, grid_cell_j, pred_label, gt_label '
        '    HAVING SUM(delta) <> 0 '
        ') '
        'INSERT INTO %s (shard_id, grid_i, grid_j, pred_label, gt_label, cnt) '
        'SELECT 0, grid_cell_i, grid_cell_j, pred_label, gt_label, net_delta '
        'FROM deltas '
        'ON CONFLICT (shard_id, grid_i, grid_j, pred_label, gt_label) '
        'DO UPDATE SET cnt = %s.cnt + EXCLUDED.cnt',
        v_pred_shard, v_pred_shard, TG_ARGV[0], TG_ARGV[0]
    );

    -- Clean up any rows that have been decremented to zero
    EXECUTE format(
        'DELETE FROM %s '
        'WHERE cnt = 0',
        TG_ARGV[0]
    );

    RETURN NULL;
END;
$body$;
"""

cur.execute(update_trigger_fn_sql)
conn.commit()
cur.execute(f"SELECT run_command_on_workers($outer${update_trigger_fn_sql}$outer$);")
conn.commit()
print("UPDATE trigger function created on coordinator and workers.")

# ---------------------------------------------------------------------------
# Step 2a: Per-shard AFTER INSERT triggers on v4_pred_patch
# ---------------------------------------------------------------------------
cur.execute(f"""
SELECT run_command_on_colocated_placements(
    '{PRED_PATCH_TABLE}',
    '{CM_TABLE}',
    $cmd$
        CREATE TRIGGER trg_update_cm AFTER INSERT ON %s
        REFERENCING NEW TABLE AS new_rows
        FOR EACH STATEMENT
        EXECUTE FUNCTION update_cm_shard(%L)
    $cmd$
);
""")
conn.commit()
print("Per-shard INSERT triggers installed on v4_pred_patch shards.")

# ---------------------------------------------------------------------------
# Step 2b: Per-shard AFTER UPDATE triggers on v4_patch
# ---------------------------------------------------------------------------
cur.execute(f"""
SELECT run_command_on_colocated_placements(
    '{PATCH_TABLE}',
    '{CM_TABLE}',
    $cmd$
        CREATE TRIGGER trg_update_cm_on_patch_update AFTER UPDATE ON %s
        REFERENCING OLD TABLE AS old_rows NEW TABLE AS new_rows
        FOR EACH STATEMENT
        EXECUTE FUNCTION update_cm_on_patch_update(%L)
    $cmd$
);
""")
conn.commit()
print("Per-shard UPDATE triggers installed on v4_patch shards.")

cur.close()
conn.close()

print("\nAll per-shard triggers installed.")

## 6 – Data generation & insertion

Two-pass insertion:
1. **Phase A** – COPY all `v4_patch` rows (no trigger overhead).
2. **Phase B** – COPY all `v4_pred_patch` rows (trigger fires per-statement,
   incrementally updating `v4_confusion_matrix_partial`).

Both passes use identical RNG seeds per worker so IDs match across tables.

In [ ]:
import numpy as np
import psycopg2
import time
from io import StringIO
from multiprocessing import Pool

# ======================
# CONFIG
# ======================
FINEST_LEVEL = 12
GRID_SIZE = 2 ** FINEST_LEVEL
NUM_LABELS = 5
TOTAL_ROWS = 1_000_000
BATCH_SIZE = 100_000
SPATIAL_CLUSTERS = 10
POINTS_PER_CLUSTER = (TOTAL_ROWS - 1_000) // SPATIAL_CLUSTERS
UNIFORM_POINTS = 1_000
WORKERS = 1


# ======================
# GRID FUNCTION (vectorized)
# ======================
def vectorized_point_to_cell(embed_x, embed_y, cell_size, level):
    scaled_size = cell_size / (2 ** level)
    i_arr = np.floor(embed_x / scaled_size).astype(np.int32)
    j_arr = np.floor(embed_y / scaled_size).astype(np.int32)
    return i_arr, j_arr


# ======================
# COPY HELPERS
# ======================
def build_patch_buffer(ids, patch_uids, gt_label):
    buf = StringIO()
    for k in range(len(ids)):
        buf.write(f"{ids[k]}\t{patch_uids[k]}\t{gt_label}\t\\N\t\\N\n")
    buf.seek(0)
    return buf


def build_pred_buffer(ids, patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py):
    buf = StringIO()
    for k in range(len(ids)):
        buf.write(
            f"{ids[k]}\t{patch_uids[k]}\t({ex[k]},{ey[k]})\t"
            f"{grid_i[k]}\t{grid_j[k]}\t{pred_labels[k]}\t({px[k]},{py[k]})\n"
        )
    buf.seek(0)
    return buf


# ======================
# BATCH GENERATORS (deterministic per worker)
# ======================
def generate_batches(worker_id):
    """Yield (ids, patch_uids, gt_label, ex, ey, grid_i, grid_j, pred_labels, px, py)
    for each batch belonging to this worker. Deterministic given worker_id."""
    local_rng = np.random.default_rng(2026 + worker_id)
    clusters_per_worker = SPATIAL_CLUSTERS // WORKERS

    # Each worker gets a full TOTAL_ROWS-wide ID range to avoid collisions.
    # Worker 0 generates extra UNIFORM_POINTS that push past TOTAL_ROWS//WORKERS.
    id_start = worker_id * TOTAL_ROWS + 1
    uid = id_start

    for cluster in range(worker_id * clusters_per_worker,
                         (worker_id + 1) * clusters_per_worker):
        mean_x = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
        mean_y = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
        std = local_rng.uniform(GRID_SIZE * 0.05, GRID_SIZE * 0.2)

        gt_label = cluster % NUM_LABELS
        pred_mode = (
            int(local_rng.integers(0, NUM_LABELS))
            if cluster % 2 == 0 else "random"
        )

        for start in range(0, POINTS_PER_CLUSTER, BATCH_SIZE):
            n = min(BATCH_SIZE, POINTS_PER_CLUSTER - start)

            ex = np.clip(local_rng.normal(mean_x, std, n), 0, GRID_SIZE)
            ey = np.clip(local_rng.normal(mean_y, std, n), 0, GRID_SIZE)

            ids = np.arange(uid, uid + n, dtype=np.int64)
            patch_uids = ids.copy()

            grid_i, grid_j = vectorized_point_to_cell(ex, ey, GRID_SIZE, FINEST_LEVEL)

            pred_labels = (
                local_rng.integers(0, NUM_LABELS, n, dtype=np.int32)
                if pred_mode == "random"
                else np.full(n, pred_mode, dtype=np.int32)
            )

            px = local_rng.uniform(0, 10_000, n)
            py = local_rng.uniform(0, 10_000, n)

            yield ids, patch_uids, gt_label, ex, ey, grid_i, grid_j, pred_labels, px, py
            uid += n

    # Uniform tail (worker 0 only)
    if worker_id == 0:
        n = UNIFORM_POINTS
        ex = local_rng.uniform(0, GRID_SIZE, n)
        ey = local_rng.uniform(0, GRID_SIZE, n)

        ids = np.arange(uid, uid + n, dtype=np.int64)
        patch_uids = ids.copy()
        grid_i, grid_j = vectorized_point_to_cell(ex, ey, GRID_SIZE, FINEST_LEVEL)
        pred_labels = local_rng.integers(0, NUM_LABELS, n, dtype=np.int32)
        px = local_rng.uniform(0, 10_000, n)
        py = local_rng.uniform(0, 10_000, n)

        yield ids, patch_uids, NUM_LABELS, ex, ey, grid_i, grid_j, pred_labels, px, py


# ======================
# WORKERS
# ======================
def worker_patch(worker_id):
    """Phase A: insert into v4_patch only."""
    conn = psycopg2.connect(DATABASE_URL)
    t0 = time.perf_counter()
    rows = 0
    try:
        for ids, patch_uids, gt_label, *_ in generate_batches(worker_id):
            buf = build_patch_buffer(ids, patch_uids, gt_label)
            cur = conn.cursor()
            cur.copy_from(
                buf, PATCH_TABLE,
                columns=("id", "patch_uid", "gt_label", "image_id", "working_mag"),
            )
            conn.commit()
            cur.close()
            rows += len(ids)
    finally:
        conn.close()
    elapsed = time.perf_counter() - t0
    print(f"[Patch W{worker_id}] {rows:,} rows in {elapsed:.2f}s "
          f"({rows/elapsed:,.0f} rows/sec)")
    return rows, elapsed


def worker_pred(worker_id):
    """Phase B: insert into v4_pred_patch (trigger fires per-statement)."""
    conn = psycopg2.connect(DATABASE_URL)
    t0 = time.perf_counter()
    rows = 0
    try:
        for ids, patch_uids, _gt, ex, ey, grid_i, grid_j, pred_labels, px, py in generate_batches(worker_id):
            buf = build_pred_buffer(ids, patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py)
            cur = conn.cursor()
            cur.copy_from(
                buf, PRED_PATCH_TABLE,
                columns=("id", "patch_uid", "embed_coords",
                         "grid_cell_i", "grid_cell_j",
                         "pred_label", "patch_coords"),
            )
            conn.commit()
            cur.close()
            rows += len(ids)
    finally:
        conn.close()
    elapsed = time.perf_counter() - t0
    print(f"[Pred  W{worker_id}] {rows:,} rows in {elapsed:.2f}s "
          f"({rows/elapsed:,.0f} rows/sec)")
    return rows, elapsed


print("Helpers defined.")

In [ ]:
t_total = time.perf_counter()

# Phase A — patch rows (no trigger)
print("=== Phase A: Inserting v4_patch rows ===")
t0 = time.perf_counter()
with Pool(WORKERS) as p:
    patch_results = p.map(worker_patch, range(WORKERS))
patch_elapsed = time.perf_counter() - t0
patch_rows = sum(r[0] for r in patch_results)
print(f"Phase A done: {patch_rows:,} rows in {patch_elapsed:.2f}s\n")

# Phase B — pred_patch rows (trigger fires per-statement)
print("=== Phase B: Inserting v4_pred_patch rows (with trigger) ===")
t0 = time.perf_counter()
with Pool(WORKERS) as p:
    pred_results = p.map(worker_pred, range(WORKERS))
pred_elapsed = time.perf_counter() - t0
pred_rows = sum(r[0] for r in pred_results)
print(f"Phase B done: {pred_rows:,} rows in {pred_elapsed:.2f}s\n")

total_elapsed = time.perf_counter() - t_total
print(f"Total insertion: {patch_rows + pred_rows:,} rows in {total_elapsed:.2f}s")
print(f"  Patch throughput:     {patch_rows/patch_elapsed:,.0f} rows/sec")
print(f"  Pred+trigger throughput: {pred_rows/pred_elapsed:,.0f} rows/sec")

## 7 – Quick sanity check on the trigger table

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"SELECT COUNT(*) FROM {CM_TABLE};")
cm_rows = cur.fetchone()[0]
print(f"{CM_TABLE} rows: {cm_rows:,}")

cur.execute(f"SELECT SUM(cnt) FROM {CM_TABLE};")
cm_total = cur.fetchone()[0]
print(f"{CM_TABLE} SUM(cnt): {cm_total:,}")

cur.execute(f"SELECT * FROM {CM_TABLE} LIMIT 10;")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

## 8 – Test UPDATE trigger on `v4_patch`

Update `gt_label` for ~1 % of patch rows (every 100th ID).  The per-shard
`AFTER UPDATE` trigger should apply negative deltas for the old label and
positive deltas for the new label in `v4_confusion_matrix_partial`.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Snapshot CM totals BEFORE the update
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(cnt)
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
cm_before = {(r[0], r[1]): r[2] for r in cur.fetchall()}
print(f"CM totals BEFORE update: {len(cm_before)} (gt,pred) pairs, "
      f"total count = {sum(cm_before.values()):,}")

# Update ~1% of patch rows: shift gt_label by +1 mod NUM_LABELS
t0 = time.perf_counter()
cur.execute(f"""
    UPDATE {PATCH_TABLE}
    SET gt_label = MOD(gt_label + 1, {NUM_LABELS})
    WHERE MOD(id, 100) = 0;
""")
updated = cur.rowcount
conn.commit()
update_elapsed = time.perf_counter() - t0
print(f"\nUpdated {updated:,} rows in {update_elapsed:.3f}s "
      f"({updated/update_elapsed:,.0f} rows/sec)")

# Snapshot CM totals AFTER the update
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(cnt)
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
cm_after = {(r[0], r[1]): r[2] for r in cur.fetchall()}
print(f"CM totals AFTER update:  {len(cm_after)} (gt,pred) pairs, "
      f"total count = {sum(cm_after.values()):,}")

# The total count should be unchanged (moved between labels, not added/removed)
before_total = sum(cm_before.values())
after_total = sum(cm_after.values())
print(f"\nTotal count preserved: {'YES' if before_total == after_total else 'NO'} "
      f"(before={before_total:,}, after={after_total:,})")

# Show which (gt,pred) pairs changed
all_keys = sorted(set(cm_before) | set(cm_after))
changed = 0
for key in all_keys:
    b = cm_before.get(key, 0)
    a = cm_after.get(key, 0)
    if b != a:
        changed += 1
        if changed <= 10:
            print(f"  gt={key[0]}, pred={key[1]}: {b:,} -> {a:,} (delta={a-b:+,})")
if changed > 10:
    print(f"  ... and {changed - 10} more changed pairs")
print(f"\n{changed} of {len(all_keys)} (gt,pred) pairs changed.")

cur.close()
conn.close()

## 9 – Verify UPDATE trigger correctness

Refresh the materialized view (which recomputes from scratch) and compare
its totals against the trigger-maintained `v4_confusion_matrix_partial`.
They should still match exactly after the `gt_label` updates.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Create the matview if it doesn't exist yet, otherwise just refresh it.
cur.execute(f"""
    SELECT COUNT(*) FROM pg_matviews WHERE matviewname = '{MATVIEW_NAME}';
""")
matview_exists = cur.fetchone()[0] > 0

t0 = time.perf_counter()
if matview_exists:
    cur.execute(f"REFRESH MATERIALIZED VIEW {MATVIEW_NAME};")
else:
    cur.execute(f"""
    CREATE MATERIALIZED VIEW {MATVIEW_NAME} AS
    SELECT
        pp.pred_label,
        COALESCE(p.gt_label, -1) AS gt_label,
        pp.grid_cell_i,
        pp.grid_cell_j,
        COUNT(*) AS patch_count
    FROM {PRED_PATCH_TABLE} pp
    LEFT JOIN {PATCH_TABLE} p ON pp.id = p.id
    GROUP BY pp.pred_label, COALESCE(p.gt_label, -1), pp.grid_cell_i, pp.grid_cell_j;
    """)
conn.commit()
matview_refresh_time = time.perf_counter() - t0
action = "Refreshed" if matview_exists else "Created"
print(f"{action} {MATVIEW_NAME} in {matview_refresh_time:.3f}s (after gt_label updates)")

# Trigger-based totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(cnt)
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
cm_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

# Matview totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(patch_count)
    FROM {MATVIEW_NAME}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
mv_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

cur.close()
conn.close()

# Compare
all_keys = sorted(set(cm_totals) | set(mv_totals))
mismatches = 0
for key in all_keys:
    cm_val = cm_totals.get(key, 0)
    mv_val = mv_totals.get(key, 0)
    if cm_val != mv_val:
        mismatches += 1
        if mismatches <= 15:
            print(f"  MISMATCH gt={key[0]}, pred={key[1]}: trigger={cm_val}, matview={mv_val}")

if mismatches == 0:
    print(f"All {len(all_keys)} (gt_label, pred_label) pairs match after UPDATE.")
else:
    print(f"\n{mismatches} mismatches out of {len(all_keys)} pairs.")

## 10 – Benchmark: query `v4_confusion_matrix_partial`

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query_cm = f"""
    SELECT gt_label, pred_label, SUM(cnt) AS total_count
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
"""

N_RUNS = 10
times_cm = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    cur.execute(query_cm)
    rows_cm = cur.fetchall()
    t1 = time.perf_counter()
    times_cm.append(t1 - t0)

cur.close()
conn.close()

times_cm_ms = [t * 1000 for t in times_cm]
print(f"=== {CM_TABLE} ===")
print(f"Rows returned: {len(rows_cm)}")
print(f"Runs: {N_RUNS}")
print(f"Min:    {min(times_cm_ms):.2f} ms")
print(f"Max:    {max(times_cm_ms):.2f} ms")
print(f"Mean:   {sum(times_cm_ms)/len(times_cm_ms):.2f} ms")
print(f"Median: {sorted(times_cm_ms)[N_RUNS//2]:.2f} ms")
print("\nSample results (first 10 rows):")
for row in rows_cm[:10]:
    print(f"  gt={row[0]}, pred={row[1]}, total={row[2]}")

## 11 – Benchmark: query materialized view

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query_mv = f"""
    SELECT gt_label, pred_label, SUM(patch_count) AS total_count
    FROM {MATVIEW_NAME}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
"""

N_RUNS = 10
times_mv = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    cur.execute(query_mv)
    rows_mv = cur.fetchall()
    t1 = time.perf_counter()
    times_mv.append(t1 - t0)

cur.close()
conn.close()

times_mv_ms = [t * 1000 for t in times_mv]
print(f"=== {MATVIEW_NAME} ===")
print(f"Rows returned: {len(rows_mv)}")
print(f"Runs: {N_RUNS}")
print(f"Min:    {min(times_mv_ms):.2f} ms")
print(f"Max:    {max(times_mv_ms):.2f} ms")
print(f"Mean:   {sum(times_mv_ms)/len(times_mv_ms):.2f} ms")
print(f"Median: {sorted(times_mv_ms)[N_RUNS//2]:.2f} ms")
print("\nSample results (first 10 rows):")
for row in rows_mv[:10]:
    print(f"  gt={row[0]}, pred={row[1]}, total={row[2]}")

## 12 – Correctness check

Compare the `(gt_label, pred_label) → total_count` from both approaches.
They should match exactly.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Trigger-based totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(cnt)
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
cm_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

# Matview totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(patch_count)
    FROM {MATVIEW_NAME}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
mv_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

cur.close()
conn.close()

# Compare
all_keys = sorted(set(cm_totals) | set(mv_totals))
mismatches = 0
for key in all_keys:
    cm_val = cm_totals.get(key, 0)
    mv_val = mv_totals.get(key, 0)
    if cm_val != mv_val:
        mismatches += 1
        print(f"  MISMATCH gt={key[0]}, pred={key[1]}: trigger={cm_val}, matview={mv_val}")

if mismatches == 0:
    print(f"All {len(all_keys)} (gt_label, pred_label) pairs match.")
else:
    print(f"\n{mismatches} mismatches out of {len(all_keys)} pairs.")

## 13 – Summary

In [ ]:
median_cm = sorted(times_cm_ms)[N_RUNS // 2]
median_mv = sorted(times_mv_ms)[N_RUNS // 2]

print(f"{'Metric':<35} {'Trigger':>12} {'Matview':>12}")
print("-" * 61)
print(f"{'Patch insert time (s)':<35} {patch_elapsed:>12.2f} {patch_elapsed:>12.2f}")
print(f"{'Pred insert time (s)':<35} {pred_elapsed:>12.2f} {'N/A':>12}")
print(f"{'gt_label UPDATE time (s)':<35} {update_elapsed:>12.3f} {'N/A':>12}")
print(f"{'Matview refresh time (s)':<35} {'N/A':>12} {matview_refresh_time:>12.3f}")
print(f"{'Query latency median (ms)':<35} {median_cm:>12.2f} {median_mv:>12.2f}")
print(f"{'(gt,pred) pairs returned':<35} {len(rows_cm):>12} {len(rows_mv):>12}")
print(f"{'Correctness mismatches':<35} {mismatches:>12} {'—':>12}")